# Phase 4 — Tuning

**Goal:** push past the 0.78229 LB target by tuning XGBoost hyperparameters and adding a `has_cabin` feature to recover cabin signal without the sparsity overfitting problem.

**Approach:** `RandomizedSearchCV` — samples random combinations from a parameter grid, faster than exhaustive grid search and often equally effective.

---

## 1. Environment

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

DATA        = Path('../data/raw')
SUBMISSIONS = Path('../submissions')
SUBMISSIONS.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_score(model, X, y):
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f'  Folds: {scores.round(4)}')
    print(f'  Mean:  {scores.mean():.4f}  Std: {scores.std():.4f}')
    return scores.mean()

print(f'train: {train.shape}   test: {test.shape}')

## 2. Feature engineering

Same pipeline as Phase 3, with one addition: `has_cabin` — a binary flag (1 if cabin was recorded, 0 if unknown). This recovers the cabin signal without the sparsity overfitting caused by 7 sparse deck dummies.

Known cabin → almost always 1st or 2nd class → higher survival. One clean feature instead of seven noisy ones.

In [ ]:
def extract_title(name_series):
    return name_series.str.split(',', n=1).str[1].str.strip().str.split('.', n=1).str[0]

TITLE_MAP = {'Mr': 'Mr', 'Mrs': 'Mrs', 'Miss': 'Miss', 'Master': 'Master'}

df = pd.concat([train, test]).reset_index(drop=True)

df['Title']     = extract_title(df['Name']).map(TITLE_MAP).fillna('Rare')
df['Fare']      = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))
df['Sex']       = df['Sex'].map({'male': 0, 'female': 1})
df['has_cabin'] = df['Cabin'].notna().astype(int)
df['Embarked']  = df['Embarked'].fillna('S')

# Age imputation
td_age = pd.get_dummies(df['Title']).drop(columns='Mr').astype(int)
fa     = pd.concat([df['Pclass'], df['Sex'], df['Fare'], td_age], axis=1)
known, unknown = fa[df['Age'].notna()], fa[df['Age'].isna()]
age_model = LinearRegression().fit(known, df.loc[df['Age'].notna(), 'Age'])
df.loc[df['Age'].isna(), 'Age'] = age_model.predict(unknown).clip(0.5, 80)

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['is_solo']    = (df['FamilySize'] == 1).astype(int)

title_dummies    = pd.get_dummies(df['Title'],    prefix='Title').drop(columns='Title_Mr').astype(int)
embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Emb').drop(columns='Emb_S').astype(int)

FEATURE_COLS = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize', 'is_solo', 'has_cabin']
df_model = pd.concat([df[FEATURE_COLS], title_dummies, embarked_dummies], axis=1)

train_mask = df['Survived'].notna()
X_train = df_model[train_mask].reset_index(drop=True)
y_train = df.loc[train_mask, 'Survived'].astype(int).reset_index(drop=True)
X_test  = df_model[~train_mask].reset_index(drop=True)

print(f'Features: {list(X_train.columns)}')
print(f'X_train: {X_train.shape}  missing: {X_train.isna().sum().sum()}')

## 3. Baseline check

Re-run the Phase 3 best model with `has_cabin` added to see if it helps before tuning.

In [ ]:
baseline = XGBClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=4,
    random_state=42, eval_metric='logloss', verbosity=0,
)

print('Baseline (100 trees) + has_cabin:')
baseline_score = cv_score(baseline, X_train, y_train)

## 4. Hyperparameter search

Key parameters to tune:
- `n_estimators` + `learning_rate` — trade-off between speed and precision of learning
- `max_depth` — tree complexity; shallow trees reduce overfitting on small datasets
- `min_child_weight` — minimum data points in a leaf; higher = more conservative splits
- `subsample` — fraction of rows used per tree; adds randomness, reduces overfitting
- `colsample_bytree` — fraction of features used per tree; same idea
- `gamma` — minimum loss reduction to make a split; acts as pruning
- `reg_alpha` / `reg_lambda` — L1/L2 regularisation on weights

In [ ]:
param_grid = {
    'n_estimators':      [100, 200, 300, 500],
    'learning_rate':     [0.01, 0.05, 0.1, 0.2],
    'max_depth':         [3, 4, 5, 6],
    'min_child_weight':  [1, 3, 5, 10],
    'subsample':         [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree':  [0.6, 0.7, 0.8, 1.0],
    'gamma':             [0, 0.1, 0.5, 1],
    'reg_alpha':         [0, 0.1, 0.5, 1],
    'reg_lambda':        [1, 2, 5, 10],
}

xgb_base = XGBClassifier(
    random_state=42, eval_metric='logloss', verbosity=0
)

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_grid,
    n_iter=100,
    scoring='accuracy',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print(f'\nBest CV score: {search.best_score_:.4f}')
print(f'Best params:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

## 5. Evaluate best model

In [ ]:
best = search.best_estimator_

print('Best model CV (re-evaluated):')
best_score = cv_score(best, X_train, y_train)

print(f'\nImprovement over baseline: {best_score - baseline_score:+.4f}')

## 6. Generate submission

In [ ]:
best.fit(X_train, y_train)
predictions = best.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived':    predictions.astype(int),
})

filename = SUBMISSIONS / f'{datetime.now().strftime("%Y-%m-%d_%H%M")}_xgboost_tuned.csv'
submission.to_csv(filename, index=False)
print(f'Saved: {filename}')
print(f'Predictions: {submission.Survived.value_counts().to_dict()}')
submission.head()

In [ ]:
# import subprocess
# result = subprocess.run([
#     'kaggle', 'competitions', 'submit',
#     '-c', 'titanic',
#     '-f', str(filename),
#     '-m', f'XGBoost tuned — CV {best_score:.4f}',
# ], capture_output=True, text=True)
# print(result.stdout or result.stderr)

---

## 7. Findings & lessons learned

**Results:**

| Model | CV | LB | Notes |
|---|---|---|---|
| XGBoost Phase 3 (100 trees, no deck) | 0.836 | 0.777 | **Best LB** |
| XGBoost tuned (RandomizedSearchCV) | 0.847 | 0.758 | Worse — hyperparameter overfitting |
| `has_cabin` alone added to Phase 3 | 0.836 | — | Zero improvement — redundant with Pclass/Fare |

**Final best submission: Phase 3 — LB 0.777**

---

**Key lessons:**

1. **Hyperparameter overfitting is real.** Running 100 search iterations × 5 folds = 500 fits effectively selects parameters that fit the specific fold arrangement, not genuine signal. The CV–LB gap grew from 0.059 (Phase 3) to 0.089 (Phase 4 tuned). More search iterations on a small dataset = more overfitting, not better generalisation.

2. **CV–LB gap is your compass.** A growing gap means the model is learning training-specific patterns. When tuning makes CV go up but LB go down, stop and simplify.

3. **Feature redundancy.** `has_cabin` added nothing — Pclass and Fare already encode the same information (cabin records existed only for 1st/2nd class passengers who paid higher fares). Creating a feature that's a proxy for an existing feature just adds noise.

4. **Small datasets have a ceiling.** With 891 training passengers, the honest ceiling for XGBoost without advanced techniques (stacking, ensembling, deep feature engineering) is around 0.78–0.80. Trying to push past it with hyperparameter search backfires.

5. **Simplicity wins on small data.** The Phase 3 model — 100 trees, sensible defaults, clean features — outperformed an aggressively tuned model. This is a recurring theme in ML: complexity is only justified when you have enough data to support it.

---

**What would be needed to beat 0.782:**
- Ensemble of multiple model types (logistic regression + random forest + XGBoost voting)
- Stacking (train a meta-model on out-of-fold predictions)
- Deeper feature interactions (e.g. Pclass × Sex × Title combined features)

These are valid next steps but the learning investment is higher and the marginal gain on this specific dataset is small.